# 03 · Degradation and confounders

In [ ]:
import json
import matplotlib.pyplot as plt
import pandas as pd

from boxbox.config import load_config
from boxbox.paths import artifact_paths, dataset_paths
from boxbox.preprocessing.build import read_lap_dataset

pd.set_option("display.width", 140)
pd.set_option("display.max_columns", 30)
plt.rcParams["figure.figsize"] = (9, 4)
COLOURS = {"SOFT": "#c93a55", "MEDIUM": "#b07c00", "HARD": "#3b6fb0", "UNKNOWN": "#77766f"}
subset, full = load_config("subset"), load_config("full")

Raw lap time mixes tyre wear with fuel burn and track evolution. Fuel correction and the causal field reference remove most of the shared trend.

In [ ]:
from boxbox.features.build import build_features
from boxbox.reporting.eda import raw_vs_corrected, degradation_curve_table
features = build_features(read_lap_dataset(subset).laps, subset)
conf = raw_vs_corrected(features)
fig, axes = plt.subplots(1, 3, figsize=(12, 3.5))
for ax, col in zip(axes, ['raw_s', 'fuel_corrected_s', 'relative_s']):
    ax.plot(conf.stint_lap, conf[col], marker='o', ms=3); ax.set(title=col, xlabel='stint lap')
fig.tight_layout()

In [ ]:
curves = degradation_curve_table(features[features.split == 'train'])
fig, ax = plt.subplots()
for compound, g in curves[curves.n >= 10].groupby('compound'):
    ax.fill_between(g.tyre_age, g.q25_s, g.q75_s, alpha=0.15, color=COLOURS[compound])
    ax.plot(g.tyre_age, g.median_s, marker='o', ms=3, color=COLOURS[compound], label=compound)
ax.set(xlabel='tyre age (laps)', ylabel='pace vs field, driver-adjusted (s)', title='Empirical degradation (train races, subset)'); ax.legend();

Linear degradation rate per compound (OLS slope of driver-adjusted pace on tyre age, train races):

In [ ]:
import numpy as np
rows = features[features.is_training_row & (features.split == 'train')]
adj = rows.y_s - rows.driver_offset_s.fillna(0)
pd.Series({c: np.polyfit(rows.tyre_age[rows.compound == c], adj[rows.compound == c], 1)[0] for c in ['SOFT', 'MEDIUM', 'HARD']}, name='s_per_lap')

Full profile: pre-2018 stints have `compound = UNKNOWN` and an inferred tyre age; the refuelling era (2005-09) has no fuel correction.

In [ ]:
full_features = build_features(read_lap_dataset(full).laps, full)
era = full_features[full_features.is_training_row].assign(adj=lambda d: d.y_s - d.driver_offset_s.fillna(0))
era.groupby(['era', era.tyre_age.clip(upper=30) // 5 * 5])['adj'].median().unstack(0).round(2)

In [ ]:
fig, ax = plt.subplots()
features[features.is_training_row]['s_deg_slope_5'].clip(-1, 1).hist(bins=60, ax=ax, color='#2a78d6')
ax.set(xlabel='causal 5-lap slope (s/lap)', title='Distribution of within-stint degradation slopes');